# Indirect PIA base-paper reproduction — increment 1

This notebook only captures the Kaggle environment and validates the supplied datasets. It does **not** install dependencies or train a model. The official implementation files remain unchanged.

In [ ]:
from pathlib import Path
import os
import subprocess

PROJECT_URL = "https://github.com/apusharma45/indirect-pia-detection.git"
PROJECT_DIR = Path("/kaggle/working/indirect-pia-detection")

if (PROJECT_DIR / ".git").is_dir():
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], check=True)
elif PROJECT_DIR.exists() and any(PROJECT_DIR.iterdir()):
    raise RuntimeError(f"Refusing to clone into non-empty directory: {PROJECT_DIR}")
else:
    subprocess.run(["git", "clone", PROJECT_URL, str(PROJECT_DIR)], check=True)

os.chdir(PROJECT_DIR)
print("Project directory:", Path.cwd())
subprocess.run(["git", "rev-parse", "HEAD"], check=True)

## Capture the untouched Kaggle environment

Run this before any package installation so we can decide whether the author's pinned environment is compatible with the assigned GPU image.

In [ ]:
subprocess.run(
    ["python", "scripts/check_environment.py", "--output", "results/environment.txt"],
    check=True,
)

## Validate the supplied data

This reads and hashes all JSON datasets and regenerates the two summaries. It does not alter any source data.

In [ ]:
subprocess.run(
    ["python", "scripts/inspect_data.py", "--data-dir", "data", "--output-dir", "results"],
    check=True,
)
print((PROJECT_DIR / "results/data_summary.md").read_text(encoding="utf-8"))

## Return the evidence through GitHub

Review the reports first. Set `PUSH_RESULTS = True` only after Git authentication is configured in Kaggle. No token is stored or printed by this notebook.

In [ ]:
RETURN_FILES = [
    "results/environment.txt",
    "results/data_summary.json",
    "results/data_summary.md",
]
subprocess.run(["git", "status", "--short", "--", *RETURN_FILES], check=True)

PUSH_RESULTS = False
if PUSH_RESULTS:
    subprocess.run(["git", "add", "--", *RETURN_FILES], check=True)
    staged = subprocess.run(["git", "diff", "--cached", "--quiet"])
    if staged.returncode == 1:
        subprocess.run(["git", "commit", "-m", "chore: record kaggle environment and data audit"], check=True)
        subprocess.run(["git", "push", "origin", "main"], check=True)
    elif staged.returncode == 0:
        print("No report changes to commit.")
    else:
        raise RuntimeError(f"git diff failed with exit code {staged.returncode}")
else:
    print("Reports are ready. Review them, configure Git authentication, then set PUSH_RESULTS=True and rerun this cell.")